# UR5e Cobot Bottom 20 Question Failure Diagnostics

This notebook groups UR5e Cobot rows by question, ranks questions by average `judge_answer_correctness_vs_ref`, and diagnoses the 20 worst-performing questions across their six replicate rows.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

RUN_DIR = ROOT / "data" / "results" / "runs" / "multi_pdf" / "2026-04-17_multi_pdf_eval"
EVAL_CSV = RUN_DIR / "merged" / "multi_pdf_eval.csv"
COMBINED_QA = RUN_DIR / "meta" / "combined_multi_pdf_qa.csv"
RAW_COBOT_QA = ROOT / "data" / "QA" / "COBOT" / "Final COBOT Lathe QA raw .csv"

df = pd.read_csv(EVAL_CSV)
qa_df = pd.read_csv(COMBINED_QA)
qa_map = qa_df.set_index("question")["machine"]
df["machine"] = df["question"].map(qa_map)
df["correct_bool"] = df["judge_answer_correctness_vs_ref"].astype(str).str.lower().map({"true": True, "false": False})

cobot_df = df[df["machine"].eq("UR5e Cobot")].copy()

question_summary = (
    cobot_df.groupby("question", dropna=False)
    .agg(
        n=("correct_bool", "size"),
        correct=("correct_bool", "sum"),
        avg_correctness=("correct_bool", "mean"),
        avg_cosine=("cosine", "mean"),
        avg_rougeL=("rougeL", "mean"),
        avg_bleu=("bleu", "mean"),
    )
    .reset_index()
    .sort_values(
        ["avg_correctness", "avg_cosine", "avg_rougeL", "avg_bleu", "question"],
        ascending=[True, True, True, True, True],
    )
)

bottom20_questions = question_summary.head(20)["question"].tolist()
raw_cobot_qa = pd.read_csv(RAW_COBOT_QA) if RAW_COBOT_QA.exists() else pd.DataFrame()

context_keywords = ("retrieved", "source", "page", "chunk", "context")
context_cols = [c for c in cobot_df.columns if any(k in c.lower() for k in context_keywords)]
richer_context_cols = [c for c in context_cols if c != "retrieved_files"]

print(f"Loaded eval CSV: {EVAL_CSV}")
print(f"UR5e Cobot rows: {len(cobot_df)}")
print(f"UR5e Cobot unique questions: {cobot_df['question'].nunique()}")
print(f"Bottom-20 questions selected after grouping by question: {len(bottom20_questions)}")
if context_cols:
    print(f"Context/retrieval columns found: {context_cols}")
else:
    print("WARNING: no retrieved/source/page/chunk/context columns found.")
if not richer_context_cols:
    print("WARNING: no source/page/chunk/context columns found beyond retrieved_files; full retrieved text is not available in this CSV.")

In [ ]:
bottom20_summary = question_summary.head(20).copy()
bottom20_summary.insert(0, "rank", range(1, len(bottom20_summary) + 1))
display(bottom20_summary[[
    "rank", "correct", "n", "avg_correctness", "avg_cosine", "avg_rougeL", "avg_bleu", "question"
]])

## UR5e Cobot Bottom 20 Failure Diagnostics

In [ ]:
def diagnose_question(question: str):
    """Show the six replicate rows and source QA metadata for one UR5e bottom-20 question."""
    matches = question_summary[question_summary["question"].eq(question)].copy()
    if matches.empty:
        raise ValueError(f"Question not found in UR5e summary: {question}")

    rank = int(question_summary.reset_index(drop=True).index[question_summary["question"].eq(question)][0]) + 1
    sub = cobot_df[cobot_df["question"].eq(question)].copy()
    sub = sub.sort_values(["approach", "model", "replicate"])

    print("=" * 120)
    print(f"Rank: {rank}")
    print(f"Question: {question}")
    print(f"Correct: {int(sub['correct_bool'].sum())}/{len(sub)}")
    print(
        "Average metrics: "
        f"cosine={sub['cosine'].mean():.4f}, "
        f"rougeL={sub['rougeL'].mean():.4f}, "
        f"bleu={sub['bleu'].mean():.4f}"
    )
    print("\nGold answer:")
    print(sub.iloc[0]["gold_answer"])

    if not raw_cobot_qa.empty:
        raw_mask = raw_cobot_qa.get("Question", pd.Series(dtype=str)).eq(question)
        if "Lora Suggestions for Revised Question Wording" in raw_cobot_qa.columns:
            raw_mask = raw_mask | raw_cobot_qa["Lora Suggestions for Revised Question Wording"].eq(question)
        raw_matches = raw_cobot_qa[raw_mask]
        print("\nOriginal COBOT QA source row:")
        if raw_matches.empty:
            print("WARNING: no matching raw COBOT QA source row found.")
        else:
            raw_cols = [
                c for c in [
                    "Question",
                    "Lora Suggestions for Revised Question Wording",
                    "Answer",
                    "Revised Question Answer",
                    "Source Page",
                    "Difficulty Type",
                    "Relation",
                ]
                if c in raw_matches.columns
            ]
            display(raw_matches[raw_cols])

    if not context_cols:
        print("\nWARNING: no retrieved/source/page/chunk/context columns found.")
    elif not richer_context_cols:
        print("\nWARNING: retrieved_files is available, but source/page/chunk/context text columns are absent.")

    diagnostic_cols = [
        "approach",
        "model",
        "replicate",
        "judge_answer_correctness_vs_ref",
        "judge_answer_helpfulness",
        "cosine",
        "rougeL",
        "bleu",
    ]
    diagnostic_cols += [c for c in context_cols if c not in diagnostic_cols]
    diagnostic_cols += ["generated_answer", "text_correctness_vs_ref"]
    diagnostic_cols = [c for c in diagnostic_cols if c in sub.columns]

    display(sub[diagnostic_cols].reset_index(names="row_index"))

### 01. What are the external connection ports on the robot?

In [ ]:
diagnose_question('What are the external connection ports on the robot?')

**Classification:** retrieval failure; generation failure; chunking/PDF parsing issue

**Diagnosis:** All six replicates retrieved the relevant control-box connection section or the full UR5e source, but the generated answers only named Mini DisplayPort and/or Ethernet plus cable-routing details. The gold answer expects the complete figure/list: Teach Pendant Port, SD card slot, Ethernet, USB 2.0, USB 3.0, Mini DisplayPort, and 10A Mini Blade Fuse. The manual text available in the parsed markdown exposes Mini DisplayPort and the fuse text, while the remaining labels appear to be embedded in an illustration. This is mainly a PDF parsing/chunking issue that left the model with incomplete text, followed by incomplete generation from the partial source.

### 02. What are the tool attachments I can put on the universal robot relating to moving objects with the arm?

In [ ]:
diagnose_question('What are the tool attachments I can put on the universal robot relating to moving objects with the arm?')

**Classification:** source ambiguity; possible gold-answer issue

**Diagnosis:** Every answer stayed close to the UR manual language: an end effector/tool attaches to the Tool Flange to manipulate a workpiece, with examples such as grippers, inspection, adhesives, and welding. The gold answer requires a detailed taxonomy of gripper types: parallel, three-finger, vacuum, magnetic, soft, and precision grippers. The raw QA row has no source page, and those detailed gripper categories are not present in the parsed UR5e manual except for a passing reference to a vacuum gripper. The failed answers are therefore mostly source-grounded refusals or generic end-effector answers, while the gold appears to rely on outside tooling knowledge.

### 03. What does the symbol with 3 arrowed lines emerging from the center of the icon mean?

In [ ]:
diagnose_question('What does the symbol with 3 arrowed lines emerging from the center of the icon mean?')

**Classification:** retrieval failure; chunking/PDF parsing issue; source ambiguity

**Diagnosis:** All six rows either said the icon could not be determined or matched a different safety/control icon. The retrieved source points to the Freedrive panel and control-icon areas, but the actual icon legend is represented as an image/table reference in the parsed manual rather than accessible text. The gold answer says the three-arrow icon is Translation control mode, allowing movement through all axes without rotation. Because the table/icon description is missing from the CSV context, the models had no reliable text anchor for the symbol.

### 04. What are the tool attachments I can put on the universal robot relating to welding?

In [ ]:
diagnose_question('What are the tool attachments I can put on the universal robot relating to welding?')

**Classification:** source ambiguity; possible gold-answer issue

**Diagnosis:** The generated answers consistently recognized that the UR manual discusses tools/end effectors and mentions welding as an application, but they did not list 'welding and arc welding attachments.' The raw QA row has no source page, and the parsed UR5e manual does not appear to enumerate welding attachment types or arc-welding attachments as a supported list. The answers are incomplete against the gold, but the gold itself looks under-supported by the provided source material.

### 05. What are the steps for moving the robot arm safely to check its position stability?

In [ ]:
diagnose_question('What are the steps for moving the robot arm safely to check its position stability?')

**Classification:** retrieval failure; generation failure

**Diagnosis:** The gold expects the Robot Arm Inspection Plan sequence: move to zero, disconnect power, inspect cables and bolts, then use Freedrive to test whether the arm holds position. Keyword and semantic runs retrieved general Move/Freedrive or Recovery Mode sections, so they missed the inspection-plan source. The long-context answers got closer and mentioned zero position, disconnecting power, and Freedrive hold testing, but still omitted the explicit cable-and-bolt inspection step. One answer also created an unsafe sequence by disconnecting power and then using Freedrive without explaining the transition.

### 06. What happens if my robot stopped in 1000ms when my stopping time safety limit is set to 200ms?

In [ ]:
diagnose_question('What happens if my robot stopped in 1000ms when my stopping time safety limit is set to 200ms?')

**Classification:** generation failure; source ambiguity

**Diagnosis:** The retrieved sections include safety functions and emergency events, but the generated answers focused on Stop Category 0, dynamic speed reduction, or uncertainty about the exact outcome. The gold answer wants the post-violation state: the robot enters Recovery Mode because the stopping-time safety limit was violated. The failure is not pure retrieval because relevant safety-limit material was often present; the model selected the stop reaction instead of the mode transition. There is also source ambiguity because the manual discusses both stop categories and recovery mode in adjacent safety contexts.

### 07. How do I lock the X axis from rotating on the arm when in freedrive mode?

In [ ]:
diagnose_question('How do I lock the X axis from rotating on the arm when in freedrive mode?')

**Classification:** chunking/PDF parsing issue; generation failure

**Diagnosis:** All rows recognized the Freedrive panel or movement-type controls but failed to name the RX toggle. Several answers recommended Translation mode, which disables rotation generally rather than locking only rotational X. The gold answer depends on a detailed Freedrive-panel axis table listing X, Y, Z, RX, RY, and RZ; the parsed markdown only exposes a table placeholder and image reference. The missing table text made the model generalize from movement modes instead of giving the specific RX-toggle instruction.

### 08. I want to control the arm using either admittance or impedance control schemes by reading joint positions/torques and commanding joint velocities/torques at a high rate. Can this be implemented using the standard hardware and controller?

In [ ]:
diagnose_question('I want to control the arm using either admittance or impedance control schemes by reading joint positions/torques and commanding joint velocities/torques at a high rate. Can this be implemented using the standard hardware and controller?')

**Classification:** source ambiguity; possible gold-answer issue; grading issue

**Diagnosis:** Every answer said the requested admittance/impedance control capability could not be determined from the provided documents. The raw COBOT QA file has no matching source row, and the parsed UR5e manual contains no 'admittance,' 'impedance,' or '125 Hz' text. The gold answer also appears internally strained: it says 'yes' with standard hardware/controller, but then says it requires direct torque control or a low-level research API. This is likely an out-of-source gold answer rather than a normal RAG failure.

### 09. How do I clean the machine?

In [ ]:
diagnose_question('How do I clean the machine?')

**Classification:** retrieval failure; generation failure; source ambiguity

**Diagnosis:** The gold cleaning procedure exists in the UR cleaning/maintenance content: remove debris, apply cleaner, agitate, dwell up to 5 minutes, rinse, and dry. However, several runs retrieved Bridgeport mill installation cleaning or mixed machine maintenance chunks, and the UR-focused answers listed approved cleaning agents and cautions but omitted the ordered procedure. The question says 'machine,' which is ambiguous in a multi-PDF run; that ambiguity helped retrieval pull non-UR machine cleaning content.

### 10. What is the risk of EMC interference, and how can it be avoided?

In [ ]:
diagnose_question('What is the risk of EMC interference, and how can it be avoided?')

**Classification:** generation failure; grading issue; source ambiguity

**Diagnosis:** The retrieved EMC section was usually relevant. Most answers captured high signal levels, unexpected behavior or damage, welding environments, and the 30 m I/O cable limit. They failed because the gold specifically requires avoiding long cables and maintaining grounding. One answer was marked correct despite also omitting grounding, while similar answers were marked false. That points to a grading inconsistency plus a source-boundary issue: the grounding text is adjacent to the EMC warning, not always part of the same concise EMC answer.

### 11. How can I set up my active program so that I only need to modify 1 or 2 variables each run?

In [ ]:
diagnose_question('How can I set up my active program so that I only need to modify 1 or 2 variables each run?')

**Classification:** retrieval failure; generation failure

**Diagnosis:** The gold answer is about Favorite program variables in the UR Run/Program variable workflow. Only the long-context mini answer produced that exact mechanism. Keyword and semantic runs retrieved Haas lathe programming, tool-nose compensation, C-axis, or general edit-mode content, then either said the method could not be determined or answered with unrelated machining features. The failed long-context nano row also drifted to Haas VPS/macros. This is a cross-machine retrieval failure followed by generation from the wrong machine context.

### 12. What mode do I need to be in to load a program?

In [ ]:
diagnose_question('What mode do I need to be in to load a program?')

**Classification:** naming mismatch; source ambiguity; possible gold-answer issue

**Diagnosis:** The raw QA row originally asked what tab is needed to load a program, with an answer about the Run tab. The revised question asks what mode is needed, and the evaluated gold answer says Automatic Mode for loading/executing and Manual Mode for modifying/creating. Several failed rows answered in terms of tabs, generic program manager behavior, or Haas LIST PROGRAM behavior. Because the question changed from 'tab' to 'mode,' the evaluation mixes UI tab terminology with operational mode terminology. The gold may be valid for operational restrictions, but it is not well aligned with the original QA source page and wording.

### 13. What are the network configuration requirements to use the network features of the robot?

In [ ]:
diagnose_question('What are the network configuration requirements to use the network features of the robot?')

**Classification:** retrieval failure; grading issue

**Diagnosis:** The gold asks for security-oriented network requirements: trusted local devices only, no inbound connections from adjacent networks, restricted outbound ports/protocols/addresses, and trusted verified URCaps/magic scripts. Keyword and semantic runs mostly retrieved network setup pages and answered with IP, DHCP, DNS, SSID, wireless, and protocol configuration. Long-context rows reached the security requirements, but grading was inconsistent: one answer missing the URCaps/magic-script item was accepted, while another very similar answer with the same omission was rejected.

### 14. I dropped some liquid on my robot arm. Will that break it or cause a safety issue?

In [ ]:
diagnose_question('I dropped some liquid on my robot arm. Will that break it or cause a safety issue?')

**Classification:** retrieval failure; source ambiguity

**Diagnosis:** The gold answer focuses on liquid risk, especially keeping liquid away from the Control Box, stopping properly, and drying after cooling. Keyword and semantic retrieval often pulled emergency events, software restrictions, or generic robot descriptions, so those answers said the risk could not be determined. Long-context answers found the wet Control Box warning and were marked correct. The question says liquid was dropped on the robot arm, while the strongest source warning is about Control Box and cables, so the failures are partly caused by retrieval and partly by ambiguous wording.

### 15. What happens if I offset the center of gravity?

In [ ]:
diagnose_question('What happens if I offset the center of gravity?')

**Classification:** generation failure; grading issue

**Diagnosis:** Most rows retrieved the relevant maximum-payload/CoG section, but the failed answers either said the effect could not be determined or used an odd condition that acceleration is reduced only if the payload CoG 'exceeds the robot's reach and payload.' They also often omitted the motion-behavior and tipping-hazard parts of the gold answer. Two similar answers were accepted even with the same awkward condition, so the main issue is generation precision with some grading inconsistency.

### 16. What are the main safety precautions that should be taken before performing maintenance on a Universal Robot Arm?

In [ ]:
diagnose_question('What are the main safety precautions that should be taken before performing maintenance on a Universal Robot Arm?')

**Classification:** generation failure; grading issue

**Diagnosis:** The retrieved maintenance section was relevant and most answers listed power-off, ESD, and dust/water precautions. The rejected rows generally failed because they did not explicitly say to discharge the Control Box, even when they warned that high voltage can remain for several hours after power-off. Similar wording was accepted in other rows, so this is partly a specificity problem in generation and partly a grading consistency problem. The gold is short and expects the exact four-item checklist.

### 17. Why do I need to set the center of gravity for the payload?

In [ ]:
diagnose_question('Why do I need to set the center of gravity for the payload?')

**Classification:** generation failure; grading issue; source ambiguity

**Diagnosis:** The gold emphasizes consequences of incorrect payload CoG: procedures such as Freedrive can fail and the arm may fall on its own. Failed rows answered with general performance reasons: rated payload, optimal motion, and acceleration adjustment. Some rows with similar performance-only framing were accepted as implicitly safe-operation related, while others were rejected for not naming Freedrive failure or falling. The source spans payload setup and maximum-payload sections, so the model often answered from a nearby but less specific source.

### 18. Why does my robot arm look to be vibrating when running?

In [ ]:
diagnose_question('Why does my robot arm look to be vibrating when running?')

**Classification:** retrieval failure

**Diagnosis:** The failures retrieved startup/initialization vibration from First Boot and treated vibration as normal brake-release behavior. The correct answer comes from the Assembly/stand-dimensioning section: inadequate dynamic stiffness or stand eigenfrequency matching robot movement can make the whole system resonate, with a minimum resonance frequency of 45 Hz. Rows that retrieved the Assembly section answered correctly. This is a clean retrieval miss rather than a bad gold answer.

### 19. I got some grease on my arm while cleaning. Is that ok?

In [ ]:
diagnose_question('I got some grease on my arm while cleaning. Is that ok?')

**Classification:** naming mismatch; source ambiguity; generation failure

**Diagnosis:** The gold answer treats 'my arm' as the user's human arm: grease is an irritant/allergen and the area should be washed with water and mild cleaner. Some failed answers interpreted 'arm' as the robot arm and said continued operation could not be determined. Others included the irritant/washing facts but started with 'Yes' or framed it as OK to continue cleaning, which conflicted with the gold's direct 'No.' The wording is ambiguous in a robot-arm dataset, and the raw QA file shows the original answer started with 'Yes' before the revised answer changed to 'No.'

### 20. What check procedures do I need to follow after modifying the robot arm?

In [ ]:
diagnose_question('What check procedures do I need to follow after modifying the robot arm?')

**Classification:** retrieval failure; generation failure; grading issue

**Diagnosis:** The gold expects the full commissioning-test checklist after modification: safety I/O connection and function tests, emergency stops and brakes, safeguard inputs/reset, reduced-mode indication, operational-mode icon, 3PE reduced-speed behavior, emergency-stop outputs, robot-moving/not-stopping/reduced outputs, and application commissioning requirements. Failed answers gave only high-level maintenance statements such as 'test all safety functions' or mixed in safety-configuration/inspection checks. One high-level answer was marked correct despite lacking the detailed list, so there is also grading inconsistency.

## Summary

The bottom 20 are not one single failure mode. The main patterns are:

- **Missing visual/table content:** connection-port labels, Freedrive icons, and RX/RY/RZ toggle details are likely trapped in images or table placeholders rather than retrievable text.
- **Cross-machine retrieval:** several keyword/semantic runs retrieved Haas or Bridgeport content for UR questions, especially cleaning, active-program variables, and program loading.
- **Gold/source problems:** gripper attachment categories, welding/arc-welding attachments, and admittance/impedance control appear weakly supported or unsupported by the parsed UR5e source. The program-loading and grease questions also show wording/revision ambiguity.
- **Grading inconsistency:** EMC grounding, network URCaps omissions, CoG answers, maintenance discharge wording, and post-modification checks were graded inconsistently across similar answers.